# Questions

$$
\begin{align*}
J(\mathbf{w}) &= \| \mathbf{y} - \mathbf{X}\mathbf{w} \|^2 \\
dJ &= d(\langle \mathbf{y} - \mathbf{X}\mathbf{w}, \mathbf{y} - \mathbf{X}\mathbf{w} \rangle) \\
   &= \langle d(\mathbf{y} - \mathbf{X}\mathbf{w}), \mathbf{y} - \mathbf{X}\mathbf{w} \rangle + \langle \mathbf{y} - \mathbf{X}\mathbf{w}, d(\mathbf{y} - \mathbf{X}\mathbf{w}) \rangle \\
   &= 2 \langle d(\mathbf{y} - \mathbf{X}\mathbf{w}), \mathbf{y} - \mathbf{X}\mathbf{w} \rangle \\
   &= 2 \langle -\mathbf{X} d\mathbf{w}, \mathbf{y} - \mathbf{X}\mathbf{w} \rangle \\
   &= 2 \langle d\mathbf{w}, -\mathbf{X}^T (\mathbf{y} - \mathbf{X}\mathbf{w}) \rangle \\
\nabla J(\mathbf{w}) &= -2\mathbf{X}^T (\mathbf{y} - \mathbf{X}\mathbf{w})
\end{align*}
$$

Градиент приравняем у нулю

$$
\begin{align*}
\nabla J(\mathbf{w}) &= -2\mathbf{X}^T(\mathbf{y} - \mathbf{X}\mathbf{w}) = 0 \\
\mathbf{X}^T\mathbf{y} - \mathbf{X}^T\mathbf{X}\mathbf{w} &= 0 \\
\mathbf{X}^T\mathbf{X}\mathbf{w} &= \mathbf{X}^T\mathbf{y} \\
\mathbf{w} &= (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}
\end{align*}
$$

Функция потерь с L2-регуляризацией:
$$
J(\mathbf{w}) = \|\mathbf{y} - \mathbf{X}\mathbf{w}\|^2 + \lambda\|\mathbf{w}\|^2
$$

Вывод оптимальных параметров:
$$
\begin{align*}
\nabla J(\mathbf{w}) &= -2\mathbf{X}^T(\mathbf{y} - \mathbf{X}\mathbf{w}) + 2\lambda\mathbf{w} = 0 \\
\mathbf{X}^T\mathbf{y} - \mathbf{X}^T\mathbf{X}\mathbf{w} + \lambda\mathbf{w} &= 0 \\
(\mathbf{X}^T\mathbf{X} + \lambda\mathbf{I})\mathbf{w} &= \mathbf{X}^T\mathbf{y} \\
\mathbf{w} &= (\mathbf{X}^T\mathbf{X} + \lambda\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}
\end{align*}
$$


Функция потерь с L1-регуляризацией (Lasso-регрессия):
$$
J(\mathbf{w}) = \|\mathbf{y} - \mathbf{X}\mathbf{w}\|^2 + \lambda\|\mathbf{w}\|_1
$$
В случае L1 внятного аналитического решения получить нельзя, L1 недеференцируема в нуле.

L1 прибавляет к весам константу противположного знака, двигая их к нулю, если вес небольшой, его сначала будет бросать возле нуля из-за противолоджного влияния градиентного спуска и L1, а потом занулит полностью 

Для нелинейных зависимостий можно использовать нелинейные преобразования, например, полиномы (PolynomialFeatures)

## Imports

In [1]:
import pandas as pd
import numpy as np
import math
import random

from copy import deepcopy

from sklearn.preprocessing import OrdinalEncoder, MultiLabelBinarizer, MinMaxScaler, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from collections import Counter

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error

## Data analysis part 2

In [2]:
train = pd.read_json('../datasets/train.json')
test = pd.read_json('../datasets/test.json')

In [3]:
upper_price_limit = np.percentile(train.price.values, 99)
lower_pirce_limit = np.percentile(train.price.values, 1)
train = train[train['price'] < upper_price_limit]
train = train[train['price'] > lower_pirce_limit]

In [4]:
upper_price_limit = np.percentile(test.price.values, 99)
lower_pirce_limit = np.percentile(test.price.values, 1)


test = test[test['price'] < upper_price_limit]
test = test[test['price'] > lower_pirce_limit]

test = test[test['bathrooms'] < 6]

test = test[test['bedrooms'] < 7]

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48343 entries, 4 to 124009
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bathrooms        48343 non-null  float64
 1   bedrooms         48343 non-null  int64  
 2   building_id      48343 non-null  object 
 3   created          48343 non-null  object 
 4   description      48343 non-null  object 
 5   display_address  48343 non-null  object 
 6   features         48343 non-null  object 
 7   latitude         48343 non-null  float64
 8   listing_id       48343 non-null  int64  
 9   longitude        48343 non-null  float64
 10  manager_id       48343 non-null  object 
 11  photos           48343 non-null  object 
 12  price            48343 non-null  int64  
 13  street_address   48343 non-null  object 
 14  interest_level   48343 non-null  object 
dtypes: float64(3), int64(3), object(9)
memory usage: 5.9+ MB


In [6]:
ord_encoder = OrdinalEncoder()

In [7]:
ord_encoder.fit(train[['interest_level']])

OrdinalEncoder()

In [8]:
train['interest_level'] = ord_encoder.transform(train[['interest_level']])

In [9]:
def clean_list(values: list):
    if isinstance(values, list):
        return [item.replace('[', '')
             .replace(']', '')
             .replace("'", '')
             .replace('"', '') 
             .replace(' ', '') for item in values]
    return values
    

In [10]:
train['features'] = train['features'].apply(clean_list)

In [11]:
list_of_features = train['features'].sum()

In [12]:
count_features = Counter(list_of_features)
most_common_features = [feature[0] for feature in count_features.most_common(20)]

In [13]:
count_features.most_common(20)

[('Elevator', 25375),
 ('HardwoodFloors', 23146),
 ('CatsAllowed', 23135),
 ('DogsAllowed', 21652),
 ('Doorman', 20479),
 ('Dishwasher', 20081),
 ('NoFee', 17793),
 ('LaundryinBuilding', 16082),
 ('FitnessCenter', 12989),
 ('Pre-War', 8971),
 ('LaundryinUnit', 8437),
 ('RoofDeck', 6417),
 ('OutdoorSpace', 5132),
 ('DiningRoom', 4890),
 ('HighSpeedInternet', 4223),
 ('Balcony', 2898),
 ('SwimmingPool', 2643),
 ('LaundryInBuilding', 2564),
 ('NewConstruction', 2504),
 ('Terrace', 2177)]

In [14]:
features_col = train.iloc[:, 6]
mlb = MultiLabelBinarizer()
encoded_features = mlb.fit_transform(features_col)
features_df = pd.DataFrame(encoded_features, columns=mlb.classes_, index=train.index)

for feature in most_common_features:
    if feature in features_df.columns:
        train[feature] = features_df[feature]
    else:
        train[feature] = 0

In [15]:
feature_list = most_common_features + ['bathrooms', 'bedrooms']

In [16]:
X_train = train[feature_list]
y_train = train['price']

In [17]:
def prepare_test():
    features_col = test.iloc[:, 6]
    mlb = MultiLabelBinarizer()
    encoded_features = mlb.fit_transform(features_col)
    features_df = pd.DataFrame(encoded_features, columns=mlb.classes_, index=test.index)

    for feature in most_common_features:
        if feature in features_df.columns:
            test[feature] = features_df[feature]
        else:
            test[feature] = 0
    
    feature_list = most_common_features + ['bathrooms', 'bedrooms']

    X_test = test[feature_list]
    y_test = test['price']

    return X_test, y_test    

In [18]:
X_test, y_test = prepare_test()

## Linear regression

In [19]:
np.random.seed(21)

### My linear regression

In [20]:
class MyLinearRegression:
    def __init__(self, method='SGD', seed=21, epoch=4, learning_rate=0.01, tol=1e-03):
        self.method = method
        self.epoch = epoch
        self.lr = learning_rate
        self.tol = tol

        np.random.seed(seed)
        random.seed(seed)
    
    def SGD(self):

        num_epoch = 0
        while num_epoch < self.epoch:
            prev_w = self.w.copy()
            indices = np.random.permutation(self.X.shape[0])
            X_shuffled = self.X[indices]
            y_shuffled = self.y[indices]
            for i in range(X_shuffled.shape[0]):
                self.w = self.w - self.lr * self.calc_grad(X_shuffled[i:i+1], y_shuffled[i:i+1], self.w)
            if np.linalg.norm(self.w - prev_w) < self.tol:
                print("Tol reached")
                return
            num_epoch += 1
        print("Num_epoch reached")

    @staticmethod
    def calc_grad(X_i:np.array, y_i, w):
        grad = -2 * X_i.T * (y_i - X_i @ w)
        return grad
        

    def fit(self, X, y):
        X = np.array(X)
        self.X = np.c_[np.ones(X.shape[0]), X]
        self.y = np.array(y).reshape(-1)
        shape_of_w = X.shape[1]
        self.w = np.random.rand(shape_of_w + 1, 1)
        self.SGD()
        return self
    
    def predict(self, X):
        X = np.array(X)
        X_b = np.c_[np.ones(X.shape[0]), X]
        return X_b.dot(self.w).ravel()

In [21]:
my_lr = MyLinearRegression()

In [22]:
my_lr.fit(X_train, y_train)

Num_epoch reached


### Comparison with sklearn version

In [23]:
result_MAE = pd.DataFrame(columns=['model', 'train', 'test'])
result_RMSE = pd.DataFrame(columns=['model', 'train', 'test'])
result_R_2 = pd.DataFrame(columns=['model', 'train', 'test'])

In [24]:
sk_lr = LinearRegression()
sk_lr.fit(X_train, y_train)

LinearRegression()

In [25]:
def calc_R2(y, preds):
    y_mean = np.mean(y)
    SST = np.sum((y - y_mean)**2)
    SSE = np.sum((y - preds)**2)
    return 1 - (SSE / SST)

In [26]:
calc_R2(y_train, my_lr.predict(X_train))

0.5143880067574691

In [27]:
def check_model_metrics(model, model_name, X_train, X_test, y_train, y_test, 
                       result_R_2, result_RMSE, result_MAE):
    # Предсказания и вычисление метрик
    train_preds = model.predict(X_train)
    train_r2 = r2_score(y_train, train_preds)
    train_rmse = root_mean_squared_error(y_train, train_preds)
    train_mae = mean_absolute_error(y_train, train_preds)
    
    test_preds = model.predict(X_test)
    test_r2 = r2_score(y_test, test_preds)
    test_rmse = root_mean_squared_error(y_test, test_preds)
    test_mae = mean_absolute_error(y_test, test_preds)
    
    # Добавление результатов в DataFrame
    new_row_r2 = {'model': model_name, 'train': train_r2, 'test': test_r2}
    new_row_rmse = {'model': model_name, 'train': train_rmse, 'test': test_rmse}
    new_row_mae = {'model': model_name, 'train': train_mae, 'test': test_mae}
    
    result_R_2 = pd.concat([result_R_2, pd.DataFrame([new_row_r2])], ignore_index=True)
    result_RMSE = pd.concat([result_RMSE, pd.DataFrame([new_row_rmse])], ignore_index=True)
    result_MAE = pd.concat([result_MAE, pd.DataFrame([new_row_mae])], ignore_index=True)

    return result_R_2, result_RMSE, result_MAE

In [28]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(my_lr, 'my_lr', X_train, X_test, y_train, y_test, 
                                      result_R_2, result_RMSE, result_MAE)

In [29]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(sk_lr, 'sk_lr', X_train, X_test, y_train, y_test, 
                                      result_R_2, result_RMSE, result_MAE)

In [30]:
result_MAE

,model,train,test
0,my_lr,726.940208,729.892212
1,sk_lr,708.737118,756.119744


In [31]:
result_R_2

,model,train,test
0,my_lr,0.514388,0.513811
1,sk_lr,0.580269,0.549580


In [32]:
result_RMSE

,model,train,test
0,my_lr,1104.945433,1096.345496
1,sk_lr,1027.262879,1055.246825


##  Ridge, Lasso, ElasticNet

### My Ridge

In [33]:
class MyRidge:
    def __init__(self, method='SGD', alpha=0.0002,  seed=21, epoch=4, learning_rate=0.01, tol=1e-03):
        self.method = method
        self.epoch = epoch
        self.lr = learning_rate
        self.tol = tol
        self.alpha = alpha

        np.random.seed(seed)
        random.seed(seed)
    
    def SGD(self):
        num_epoch = 0
        while num_epoch < self.epoch:
            prev_w = self.w.copy()
            indices = np.random.permutation(self.X.shape[0])
            X_shuffled = self.X[indices]
            y_shuffled = self.y[indices]
            for i in range(X_shuffled.shape[0]):
                self.w = self.w - self.lr * self.calc_grad(X_shuffled[i:i+1], y_shuffled[i:i+1], self.w, self.alpha)
            if np.linalg.norm(self.w - prev_w) < self.tol:
                print("Tol reached")
                return
            num_epoch += 1
        print("Num_epoch reached")

    @staticmethod
    def calc_grad(X_i:np.array, y_i, w, alpha):
        reg_term = 2 * alpha * np.vstack([0, w[1:]]) 
        grad = -2 * X_i.T @ (y_i - X_i @ w) + reg_term
        return grad
        

    def fit(self, X, y):
        X = np.array(X)
        self.X = np.c_[np.ones(X.shape[0]), X]
        self.y = np.array(y).reshape(-1)
        shape_of_w = X.shape[1]
        self.w = np.random.rand(shape_of_w + 1, 1)
        self.SGD()
        return self
    
    def predict(self, X):
        X = np.array(X)
        X_b = np.c_[np.ones(X.shape[0]), X]
        return X_b.dot(self.w).ravel()

In [34]:
my_ridge = MyRidge()

In [35]:
my_ridge.fit(X_train, y_train)

Num_epoch reached


In [36]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(my_ridge, 'my_ridge', X_train, X_test, y_train, y_test, 
                                      result_R_2, result_RMSE, result_MAE)

### Sklearn Ridge

In [37]:
sk_ridge = Ridge()

In [38]:
sk_ridge.fit(X_train, y_train)

Ridge()

In [39]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(sk_ridge, 'sk_ridge', X_train, X_test, y_train, y_test, 
                                      result_R_2, result_RMSE, result_MAE)

## My Lasso

In [40]:
class MyLasso:
    def __init__(self, method='SGD', alpha=0.00002,  seed=21, epoch=4, learning_rate=0.01, tol=1e-03):
        self.method = method
        self.epoch = epoch
        self.lr = learning_rate
        self.tol = tol
        self.alpha = alpha

        np.random.seed(seed)
        random.seed(seed)
    
    def SGD(self):
        num_epoch = 0
        while num_epoch < self.epoch:
            prev_w = self.w.copy()
            indices = np.random.permutation(self.X.shape[0])
            X_shuffled = self.X[indices]
            y_shuffled = self.y[indices]
            for i in range(X_shuffled.shape[0]):
                self.w = self.w - self.lr * self.calc_grad(X_shuffled[i:i+1], y_shuffled[i:i+1], self.w, self.alpha)
            if np.linalg.norm(self.w - prev_w) < self.tol:
                print("Tol reached")
                return
            num_epoch += 1
        print("Num_epoch reached")

    @staticmethod
    def calc_grad(X_i:np.array, y_i, w, alpha):
        reg_term = alpha * np.sign(np.vstack([0, w[1:]])) 
        grad = -2 * X_i.T @ (y_i - X_i @ w) + reg_term
        return grad
         

    def fit(self, X, y):
        X = np.array(X)
        self.X = np.c_[np.ones(X.shape[0]), X]
        self.y = np.array(y).reshape(-1)
        shape_of_w = X.shape[1]
        self.w = np.random.rand(shape_of_w + 1, 1)
        self.SGD()
        return self
    
    def predict(self, X):
        X = np.array(X)
        X_b = np.c_[np.ones(X.shape[0]), X]
        return X_b.dot(self.w).ravel()

In [41]:
my_lasso = MyLasso()

In [42]:
my_lasso.fit(X_train, y_train)

Num_epoch reached


In [43]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(my_lasso, 'my_lasso', X_train, X_test, y_train, y_test, 
                                      result_R_2, result_RMSE, result_MAE)

### Sklearn Lasso

In [44]:
sk_lasso = Lasso()

In [45]:
sk_lasso.fit(X_train, y_train)

Lasso()

In [46]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(sk_lasso, 'sk_lasso', X_train, X_test, y_train, y_test, 
                                      result_R_2, result_RMSE, result_MAE)

## My ElasticNet

In [47]:
class MyElasticNet:
    def __init__(self, method='SGD', alpha=1.0, l1_ratio=0.5,  seed=21, epoch=4, learning_rate=0.01, tol=1e-03):
        self.method = method
        self.epoch = epoch
        self.lr = learning_rate
        self.tol = tol
        self.alpha = alpha
        self.l1_ratio = l1_ratio

        np.random.seed(seed)
        random.seed(seed)
    
    def SGD(self):
        num_epoch = 0
        while num_epoch < self.epoch:
            prev_w = self.w.copy()
            indices = np.random.permutation(self.X.shape[0])
            X_shuffled = self.X[indices]
            y_shuffled = self.y[indices]
            for i in range(X_shuffled.shape[0]):
                self.w = self.w - self.lr * self.calc_grad(X_shuffled[i:i+1], y_shuffled[i:i+1], self.w, self.alpha, self.l1_ratio)
            if np.linalg.norm(self.w - prev_w) < self.tol:
                print("Tol reached")
                return
            num_epoch += 1
        print("Num_epoch reached")

    @staticmethod
    def calc_grad(X_i:np.array, y_i, w, alpha, l1_ratio):
        grad = -2 * X_i.T * (y_i - X_i @ w) + alpha * l1_ratio * np.sign(w) + alpha * (1 - l1_ratio) * w
        return grad
         

    def fit(self, X, y):
        X = np.array(X)
        self.X = np.c_[np.ones(X.shape[0]), X]
        self.y = np.array(y).reshape(-1)
        shape_of_w = X.shape[1]
        self.w = np.random.rand(shape_of_w + 1, 1)
        self.SGD()
        return self
    
    def predict(self, X):
        X = np.array(X)
        X_b = np.c_[np.ones(X.shape[0]), X]
        return X_b.dot(self.w).ravel()

In [48]:
my_elnet = MyElasticNet()

In [49]:
my_elnet.fit(X_train, y_train)

Num_epoch reached


In [50]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(my_elnet, 'my_elnet', X_train, X_test, y_train, y_test, 
                                      result_R_2, result_RMSE, result_MAE)

### Sklearn ElasticNet

In [51]:
sk_elnet = ElasticNet()

In [52]:
sk_elnet.fit(X_train, y_train)

ElasticNet()

In [53]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(sk_elnet, 'sk_elnet', X_train, X_test, y_train, y_test, 
                                      result_R_2, result_RMSE, result_MAE)

### Comprs

In [54]:
result_RMSE

,model,train,test
0,my_lr,1104.945433,1096.345496
1,sk_lr,1027.262879,1055.246825
2,my_ridge,1104.928288,1096.470064
3,sk_ridge,1027.262884,1055.242189
4,my_lasso,1104.945418,1096.345513
5,sk_lasso,1027.454730,1053.834690
6,my_elnet,1170.082125,1275.445119
7,sk_elnet,1180.302091,1202.062009


In [55]:
result_MAE

,model,train,test
0,my_lr,726.940208,729.892212
1,sk_lr,708.737118,756.119744
2,my_ridge,726.858517,729.899260
3,sk_ridge,708.734175,756.105194
4,my_lasso,726.940190,729.892210
5,sk_lasso,708.383322,752.888366
6,my_elnet,776.717480,865.594952
7,sk_elnet,804.298273,802.429152


In [56]:
result_R_2

,model,train,test
0,my_lr,0.514388,0.513811
1,sk_lr,0.580269,0.549580
2,my_ridge,0.514403,0.513701
3,sk_ridge,0.580269,0.549584
4,my_lasso,0.514388,0.513811
5,sk_lasso,0.580112,0.550784
6,my_elnet,0.455447,0.341989
7,sk_elnet,0.445892,0.415528


Результаты довольно близкие

## Feature normalization

Нормализация обязательна:
- В алгоритмах основанных на градиентах(Линейная регрессия, Нейросети)
- В алгоритмах, основанных на расстояниях(Kmeans++)
В численно большие признакие будут иметь большее влияние, что не обязательно правильно.

Когда нормализация необзательна:
- Во всех остальных случаях(деревья и прочее)

### MinMaxScaler и StandartScaler

Min_max:

$$
x_{\text{scaled}} = \frac{x - \min_{\text{original}}}{\max_{\text{original}} - \min_{\text{original}}}
$$

Standart:

$$
x_{\text{scaled}} = \frac{x - \mu}{\sigma}
$$

In [57]:
def train_and_evaluate_models(X_train, X_test, y_train, y_test, 
                              result_R_2, result_RMSE, result_MAE,
                              ridge_alpha=1.0, lasso_alpha=1.0, 
                              elastic_alpha=1.0, elastic_l1_ratio=0.5,
                              scaler = 'MinMaxScaler',
                              postfix = 'MinMaxScaler'):
    
    if scaler == 'MinMaxScaler':
        scaler = MinMaxScaler()
        scaler.fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)
    
    if scaler == 'StandartScaler':
        scaler = StandardScaler()
        scaler.fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    models = {
        'Linear Regression': LinearRegression(),
        'Ridge': Ridge(alpha=ridge_alpha),
        'Lasso': Lasso(alpha=lasso_alpha),
        'ElasticNet': ElasticNet(alpha=elastic_alpha, l1_ratio=elastic_l1_ratio)
    }
    
    for name, model in models.items():
        print(f"Обучение модели: {name}")
        model.fit(X_train, y_train)
        result_R_2, result_RMSE, result_MAE = check_model_metrics(
            model, name + ' ' + postfix, X_train, X_test, y_train, y_test, 
            result_R_2, result_RMSE, result_MAE
        )
    
    return result_R_2, result_RMSE, result_MAE

In [58]:
def min_max_scale(X, feature_range=(0, 1)):

    X = np.asarray(X)
    a, b = feature_range
    
    X_min = np.min(X, axis=0)
    X_max = np.max(X, axis=0)
    
    range_ = X_max - X_min
    range_[range_ == 0] = 1  
    
    X_scaled = a + (X - X_min) * (b - a) / range_
    
    return X_scaled

In [59]:
def standart_scale(X):
    X = np.asarray(X)
    x_mean = np.mean(X, axis=0)
    x_std = np.std(X, axis=0, ddof=1)
    x_std[x_std == 0] = 1
    X_scaled = (X - x_mean)/ x_std
    return X_scaled

In [60]:
sk_min_max = MinMaxScaler()
sk_min_max.fit(X_train)
X_train_min_max = sk_min_max.transform(X_train)

In [61]:
sk_standart = StandardScaler()
sk_standart.fit(X_train)
X_train_standart = sk_standart.transform(X_train)

In [62]:
display(min_max_scale(X_train)[0], X_train_min_max[0]) 

array([0.   , 1.   , 1.   , 1.   , 0.   , 1.   , 0.   , 1.   , 0.   ,
       1.   , 0.   , 0.   , 0.   , 1.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.1  , 0.125])

array([0.   , 1.   , 1.   , 1.   , 0.   , 1.   , 0.   , 1.   , 0.   ,
       1.   , 0.   , 0.   , 0.   , 1.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.1  , 0.125])

In [63]:
display(standart_scale(X_train)[0], X_train_standart[0])

array([-1.05108284,  1.04335464,  1.04383043,  1.11027081, -0.85729077,
        1.18632665, -0.76315805,  1.4163298 , -0.60612764,  2.09492663,
       -0.45980155, -0.39121889, -0.34462095,  2.9809244 , -0.30937719,
       -0.25252341, -0.24048389, -0.2366581 , -0.23371958, -0.21715188,
       -0.42728464, -0.48507028])

array([-1.05109371,  1.04336543,  1.04384122,  1.11028229, -0.85729964,
        1.18633892, -0.76316594,  1.41634445, -0.60613391,  2.0949483 ,
       -0.45980631, -0.39122293, -0.34462451,  2.98095523, -0.30938039,
       -0.25252602, -0.24048638, -0.23666054, -0.233722  , -0.21715413,
       -0.42728906, -0.4850753 ])

In [64]:
result_R_2, result_RMSE, result_MAE = train_and_evaluate_models(X_train, 
                                                                X_test, 
                                                                y_train, 
                                                                y_test, 
                                                                result_R_2, result_RMSE, result_MAE,
                                                                ridge_alpha=1.0, lasso_alpha=1.0, 
                                                                elastic_alpha=1.0, elastic_l1_ratio=0.5,)

Обучение модели: Linear Regression
Обучение модели: Ridge
Обучение модели: Lasso
Обучение модели: ElasticNet


In [65]:
result_R_2, result_RMSE, result_MAE = train_and_evaluate_models(X_train, 
                                                                X_test, 
                                                                y_train, 
                                                                y_test, 
                                                                result_R_2, result_RMSE, result_MAE,
                                                                ridge_alpha=1.0, lasso_alpha=1.0, 
                                                                elastic_alpha=1.0, elastic_l1_ratio=0.5,
                                                                postfix='StandardScaler',
                                                                scaler='StandardScaler')

Обучение модели: Linear Regression
Обучение модели: Ridge
Обучение модели: Lasso
Обучение модели: ElasticNet


In [66]:
pf = PolynomialFeatures(degree=10).fit(X_train[['bathrooms', 'bedrooms']])

In [67]:
X_train_poly = deepcopy(X_train)
X_train_poly[pf.get_feature_names_out()] = pf.transform(X_train[['bathrooms', 'bedrooms']])
X_train_poly = X_train_poly.drop(['bathrooms', 'bedrooms'], axis=1)

In [68]:
X_test_poly = deepcopy(X_test)
X_test_poly[pf.get_feature_names_out()] = pf.transform(X_test[['bathrooms', 'bedrooms']])
X_test_poly = X_test_poly.drop(['bathrooms', 'bedrooms'], axis=1)

In [69]:
result_R_2, result_RMSE, result_MAE = train_and_evaluate_models(X_train_poly, 
                                                                X_test_poly, 
                                                                y_train, 
                                                                y_test, 
                                                                result_R_2, result_RMSE, result_MAE,
                                                                ridge_alpha=1.0, lasso_alpha=1.0, 
                                                                elastic_alpha=1.0, elastic_l1_ratio=0.5,
                                                                postfix='Poly')

Обучение модели: Linear Regression
Обучение модели: Ridge
Обучение модели: Lasso
Обучение модели: ElasticNet


In [70]:
display(result_R_2, result_RMSE, result_MAE)

,model,train,test
0,my_lr,0.514388,0.513811
1,sk_lr,0.580269,0.549580
2,my_ridge,0.514403,0.513701
3,sk_ridge,0.580269,0.549584
4,my_lasso,0.514388,0.513811
5,sk_lasso,0.580112,0.550784
6,my_elnet,0.455447,0.341989
7,sk_elnet,0.445892,0.415528
8,Linear Regression MinMaxScaler,0.580269,0.549580
9,Ridge MinMaxScaler,0.580243,0.549345


,model,train,test
0,my_lr,1104.945433,1096.345496
1,sk_lr,1027.262879,1055.246825
2,my_ridge,1104.928288,1096.470064
3,sk_ridge,1027.262884,1055.242189
4,my_lasso,1104.945418,1096.345513
5,sk_lasso,1027.454730,1053.834690
6,my_elnet,1170.082125,1275.445119
7,sk_elnet,1180.302091,1202.062009
8,Linear Regression MinMaxScaler,1027.262879,1055.246825
9,Ridge MinMaxScaler,1027.295187,1055.521605


,model,train,test
0,my_lr,726.940208,729.892212
1,sk_lr,708.737118,756.119744
2,my_ridge,726.858517,729.899260
3,sk_ridge,708.734175,756.105194
4,my_lasso,726.940190,729.892210
5,sk_lasso,708.383322,752.888366
6,my_elnet,776.717480,865.594952
7,sk_elnet,804.298273,802.429152
8,Linear Regression MinMaxScaler,708.737118,756.119744
9,Ridge MinMaxScaler,708.827934,756.273580


## Naive

In [71]:
naive_train_mean_mae = mean_absolute_error(y_train, [y_train.mean()] * len(y_train))
naive_train_mean_rmse = root_mean_squared_error(y_train, [y_train.mean()] * len(y_train))
display(naive_train_mean_mae, naive_train_mean_rmse)

1134.3614729178958

1585.609098964619

In [72]:
naive_train_median_mae = mean_absolute_error(y_train, [y_train.median()] * len(y_train))
naive_train_median_rmse = root_mean_squared_error(y_train, [y_train.median()] * len(y_train))
display(naive_train_median_mae, naive_train_median_rmse)

1082.2208592764205

1631.7199525081228

In [73]:
naive_test_mean_mae = mean_absolute_error(y_test, [y_test.mean()] * len(y_test))
naive_test_mean_rmse = root_mean_squared_error(y_test, [y_test.mean()] * len(y_test))
display(naive_test_mean_mae, naive_test_mean_rmse)

1128.406368238796

1572.3348689618651

In [74]:
naive_test_median_mae = mean_absolute_error(y_test, [y_test.median()] * len(y_test))
naive_test_median_rmse = root_mean_squared_error(y_test, [y_test.median()] * len(y_test))
display(naive_test_median_mae, naive_test_median_rmse)

1078.26573397883

1616.9418671566937

In [75]:
result_MAE = pd.concat(
    [result_MAE, pd.DataFrame([{'model': 'mean',
                                'train': naive_train_mean_mae ,
                                'test': naive_test_mean_mae }])],
    ignore_index=True  
)

result_MAE = pd.concat(
    [result_MAE, pd.DataFrame([{'model': 'median',
                                'train': naive_train_median_mae ,
                                'test': naive_test_median_mae }])],
    ignore_index=True  
)

result_RMSE = pd.concat(
    [result_RMSE, pd.DataFrame([{'model': 'mean',
                                'train': naive_train_mean_rmse ,
                                'test': naive_test_mean_rmse }])],
    ignore_index=True  
)

result_RMSE = pd.concat(
    [result_RMSE, pd.DataFrame([{'model': 'median',
                                'train': naive_train_median_rmse ,
                                'test': naive_test_median_rmse}])],
    ignore_index=True  
)

## Results

In [76]:
display(result_R_2, result_RMSE, result_MAE)

,model,train,test
0,my_lr,0.514388,0.513811
1,sk_lr,0.580269,0.549580
2,my_ridge,0.514403,0.513701
3,sk_ridge,0.580269,0.549584
4,my_lasso,0.514388,0.513811
5,sk_lasso,0.580112,0.550784
6,my_elnet,0.455447,0.341989
7,sk_elnet,0.445892,0.415528
8,Linear Regression MinMaxScaler,0.580269,0.549580
9,Ridge MinMaxScaler,0.580243,0.549345


,model,train,test
0,my_lr,1104.945433,1096.345496
1,sk_lr,1027.262879,1055.246825
2,my_ridge,1104.928288,1096.470064
3,sk_ridge,1027.262884,1055.242189
4,my_lasso,1104.945418,1096.345513
5,sk_lasso,1027.454730,1053.834690
6,my_elnet,1170.082125,1275.445119
7,sk_elnet,1180.302091,1202.062009
8,Linear Regression MinMaxScaler,1027.262879,1055.246825
9,Ridge MinMaxScaler,1027.295187,1055.521605


,model,train,test
0,my_lr,726.940208,729.892212
1,sk_lr,708.737118,756.119744
2,my_ridge,726.858517,729.899260
3,sk_ridge,708.734175,756.105194
4,my_lasso,726.940190,729.892210
5,sk_lasso,708.383322,752.888366
6,my_elnet,776.717480,865.594952
7,sk_elnet,804.298273,802.429152
8,Linear Regression MinMaxScaler,708.737118,756.119744
9,Ridge MinMaxScaler,708.827934,756.273580
